# 04. Fine-Tuning y Comparativa de Fashion-CLIP vs Adaptador Estético

Este notebook implementa el pipeline completo de Fine-Tuning de compatibilidad estética (historias , , , , , , ).

### Estrategia:
1. **Backbone Congelado:** Se mantiene el extractor visual de  intacto para preservar el conocimiento previo de moda sin catastrophic forgetting.
2. **Adaptador de Compatibilidad D=128:** Se entrena una cabeza densa ligera con Contrastive Loss (InfoNCE) sobre pares de prendas y clasificación sobre 12 estéticas ampliadas.
3. **Calibración de Temperatura:** Se ajusta el parámetro $ para obtener probabilidades calibradas.
4. **Comparativa Lado a Lado:** Se mide el Baseline Zero-Shot vs el Adaptador en F1-Macro y FITB (Fill-In-The-Blank, meta >= 70%), registrando ambos experimentos en DagsHub / MLflow.

In [ ]:
import platform
import sys
from pathlib import Path

# Deteccion automatica de entorno (Colab vs Local)
IS_COLAB = Path("/content").exists()

if IS_COLAB:
    REPO_ROOT = Path("/content/MACHINE-BULLS") if Path("/content/MACHINE-BULLS").exists() else Path("/content")
    from google.colab import drive
    drive.mount("/content/drive")
    print("[OK] Entorno Google Colab detectado")
else:
    REPO_ROOT = Path.cwd().resolve()
    for parent in [REPO_ROOT, *REPO_ROOT.parents]:
        if (parent / "notebooks" / "04_clip_fine_tuning.ipynb").exists() or (parent / "sample_data").exists():
            REPO_ROOT = parent
            break
    print(f"[OK] Entorno Local detectado | Repo: {REPO_ROOT}")

print(f"Python: {sys.version.split()[0]} | Plataforma: {platform.platform()}")

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env")
load_dotenv(REPO_ROOT / "backend" / ".env")

# Configuración de secretos para DagsHub y MLflow
token = os.environ.get("DAGSHUB_USER_TOKEN") or os.environ.get("DAGSHUB_TOKEN")
if token:
    os.environ["DAGSHUB_TOKEN"] = token

import dagshub
import mlflow

try:
    dagshub.init(repo_owner="jesusrodrigov", repo_name="machine-bulls", mlflow=True)
    print("[OK] Conectado exitosamente a DagsHub MLflow")
except Exception as ex:
    print(f"[INFO] MLflow local: {ex}")

In [ ]:
# Asegurar que el modulo src de backend este en el path
backend_src = REPO_ROOT / "backend" / "src"
if str(backend_src) not in sys.path:
    sys.path.insert(0, str(backend_src))

from ml.augmentation import verificar_splits_anti_fuga, get_image_transforms
from ml.models.adapter import AestheticProjectionHead
from ml.models.prompts import ESTETICAS_AMPLIADAS
from ml.validation.evaluate_fitb import evaluate_fitb
from ml.validation.temperature_scaling import ModelWithTemperature
from ml.validation.run_comparison import run_baseline_evaluation, run_adapter_evaluation, print_comparison_table

print("[OK] Modulos de StyleSync importados correctamente.")
print(f"[INFO] Catálogo ampliado de estéticas ({len(ESTETICAS_AMPLIADAS)}): {list(ESTETICAS_AMPLIADAS)}")

In [ ]:
# Ejecutar comparativa formal completa
b_metrics = run_baseline_evaluation()
a_metrics = run_adapter_evaluation()
print_comparison_table(b_metrics, a_metrics)